In [88]:
import os
import requests
import numpy as np
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from fastapi import FastAPI, Header, HTTPException

In [89]:
load_dotenv(override=True)

True

# Menampilkan semua data sales order yang telah di ubah menjadi format document

In [90]:
def req_so_data_as_a_list_document() -> list[Document]:
    params = {
        "scopes": "filterRespo",
        "detail": "true",
        "join": "true",
        "paginate": 250
    }
    headers = {
        "Authorization": f"Bearer {os.getenv('T_SO_BEARER_TOKEN')}",
        "Accept": "application/json",
    }
    response = requests.get(
        os.getenv("T_SO_ENDPOINT"),
        headers=headers,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    data = response.json().get("data", [])
    return [
        Document(
            page_content=(
                f"Business Unit: {so.get('comp.name')}, "
                f"Sub Business Unit: {so.get('sub_comp.name')}, "
                f"Cabang: {so.get('branch.name')}, "
                f"No. TRX SO: {so.get('no')}, "
                f"SO Date: {so.get('date')}, "
                f"SO Type: {so.get('type')}, "
                f"Item Type: {so.get('item_type.value1')}, "
                f"Customer Type: {so.get('cust_type.value1')}, "
                f"Pemasaran: {so.get('pemasaran.value1')}, "
                f"Pembayaran: {so.get('pembayaran.value1')}, "
                f"Customer: {so.get('cust_name')}, "
                f"Alamat Customer: {so.get('cust_addr')}, "
                f"No. NPWP: {so.get('cust_npwp')}, "
                f"Nama NPWP: {so.get('cust_npwpname')}, "
                f"Contact Person: {so.get('cust_cp')}, "
                f"No. PO Customer: {so.get('cust_no_po')}, "
                f"Bill To: {so.get('ship.name')}, "
                f"Bill To Address: {so.get('ship.addr')}, "
                f"Order Type 1: {so.get('order_type1.value1')}, "
                f"Order Type 2: {so.get('order_type2.value1')}, "
                f"PPN Type: {so.get('ppn_type')}, "
                f"SO Prospek: {so.get('is_prospek')}, "
                f"Nama Penerima: {so.get('ship.name')}, "
                f"Alamat: {so.get('ship.addr')}, "
                f"Due Date: {so.get('ship_duedate')}, "
                f"Payment Term: {so.get('pay_term.value1')}, "
                f"Project: {so.get('project')}, "
                f"Down Payment Percentage: {so.get('dp_pct')}, "
                f"Down Payment Amount: {so.get('dp_amt')}, "
                f"Currency: {so.get('currency.name')}, "
                f"Exchange Rate: {so.get('currency_rate')}, "
                f"Catatan: {so.get('note')}, "
                f"Status: {so.get('status.value1')}, "
                f"Est. Ekspedisi Percentage: {so.get('exp_pct')}, "
                f"Est. Ekspedisi Amount: {so.get('exp_amt')}, "
                f"Est. Operasional Percentage: {so.get('ops_pct')}, "
                f"Est. Operasional Amount: {so.get('ops_amt')}, "
                f"Total Amount: {so.get('amt')}, "
                f"Total Discount Amount: {so.get('disc_amt')}, "
                f"DPP: {so.get('dpp')}, "
                f"PPN Percent: {so.get('ppn_pct')}, "
                f"PPN Amount: {so.get('ppn_amt')}, "
                f"Grand Total: {so.get('netto')}"
            )
        )
        for so in data
    ]

so_data_as_document = req_so_data_as_a_list_document()
so_data_as_document

[Document(metadata={}, page_content='Business Unit: PENERBITAN, Sub Business Unit: Publishing JPBOOK, Cabang: JPBOOK Surabaya, No. TRX SO: SO-BR0011-2605-00000006, SO Date: 13/05/2026, SO Type: PUBLISHING, Item Type: FINISH GOOD, Customer Type: KOMERSIAL (B2B), Pemasaran: DIRECT (TANPA PERANTARA), Pembayaran: TEMPO, Customer: UPT SD N 01 GUNUNG CAHAYA, Alamat Customer: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung, No. NPWP: None, Nama NPWP: UPT SD N 01 GUNUNG CAHAYA, Contact Person: None, No. PO Customer: None, Bill To: UPT SD N 01 GUNUNG CAHAYA, Bill To Address: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung , Order Type 1: ONLINE, Order Type 2: SIPLAH, PPN Type: EXCLUDE, SO Prospek: False, Nama Penerima: UPT SD N 01 GUNUNG CAHAYA, Alamat: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung , Due Date: 27/05/2026, Payment Term: 120 HARI, Project: None, Down Payment Percentage: None, Down Payment Amo

# Menampilkan contoh hasil embedding salah satu data sales order

In [91]:
print("contoh salah satu data sales oder yang telah di format menjadi document: ")
print(so_data_as_document[0])
print("\n=================================================================\n")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(
    [so_data_as_document[0]], 
    embedding_model
) 
embeded_so_data = np.array(vectorstore.index.reconstruct(0))
print("\n=================================================================\n")
print("contoh hasil embedding salah satu data sales oder yang telah di format menjadi document: ")
print(embeded_so_data)

contoh salah satu data sales oder yang telah di format menjadi document: 
page_content='Business Unit: PENERBITAN, Sub Business Unit: Publishing JPBOOK, Cabang: JPBOOK Surabaya, No. TRX SO: SO-BR0011-2605-00000006, SO Date: 13/05/2026, SO Type: PUBLISHING, Item Type: FINISH GOOD, Customer Type: KOMERSIAL (B2B), Pemasaran: DIRECT (TANPA PERANTARA), Pembayaran: TEMPO, Customer: UPT SD N 01 GUNUNG CAHAYA, Alamat Customer: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung, No. NPWP: None, Nama NPWP: UPT SD N 01 GUNUNG CAHAYA, Contact Person: None, No. PO Customer: None, Bill To: UPT SD N 01 GUNUNG CAHAYA, Bill To Address: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung , Order Type 1: ONLINE, Order Type 2: SIPLAH, PPN Type: EXCLUDE, SO Prospek: False, Nama Penerima: UPT SD N 01 GUNUNG CAHAYA, Alamat: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung , Due Date: 27/05/2026, Payment Term: 120 HARI, Project: No

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7404.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




contoh hasil embedding salah satu data sales oder yang telah di format menjadi document: 
[-5.98610975e-02  2.26771813e-02 -5.22638299e-02 -4.42464612e-02
 -7.30603486e-02  3.46487551e-03  5.20276539e-02  2.31483439e-03
  4.23586294e-02 -7.65899010e-03  1.54875621e-01  1.69887133e-02
  1.72488056e-02 -7.63070732e-02  2.11508609e-02 -5.38017526e-02
 -2.35363133e-02 -2.44172290e-02 -3.95992771e-02 -5.01720011e-02
 -9.14537907e-03  1.82744563e-02 -2.28466373e-03 -4.79206331e-02
 -5.02947792e-02 -6.75499737e-02 -1.59274391e-03  4.64972705e-02
 -3.96387056e-02 -1.29550353e-01 -1.35139013e-02  1.42607033e-01
  2.78772716e-03 -2.97290254e-02  1.47008613e-01  2.16433443e-02
 -6.29875958e-02  4.34040539e-02  6.38608187e-02  1.34340497e-02
 -1.14104170e-02 -3.80805694e-02 -3.42979170e-02 -3.38549763e-02
  2.63084117e-02 -2.20187940e-02 -6.33005351e-02  1.08274501e-02
 -1.10768084e-03  4.11784090e-02 -1.14090167e-01  4.12948132e-02
 -2.92281937e-02  4.50310707e-02  2.19770391e-02 -9.33310315e-0

# Menampilkan contoh query dan hasil embedding query

In [92]:
embeded_query = embedding_model.embed_query("Siapa nama customer pada data sales order dengan No. TRX SO: SO-BR0011-2605-00000006?")
print("Siapa nama customer pada data sales order dengan No. TRX SO: SO-BR0011-2605-00000006?")
print(embeded_query)

Siapa nama customer pada data sales order dengan No. TRX SO: SO-BR0011-2605-00000006?
[-0.10211622714996338, 0.02517821453511715, -0.039129339158535004, -0.04913618415594101, -0.11129520833492279, -0.027086937800049782, 0.04770001769065857, 0.016120124608278275, 0.028990212827920914, -0.018244776874780655, 0.11045758426189423, 0.04905102401971817, -0.03385816887021065, -0.1092018112540245, -0.03643539920449257, -0.06257018446922302, -0.042251162230968475, -0.08245939016342163, -0.05062417313456535, 0.0070544336922466755, 0.0019382309401407838, 0.03141807019710541, -0.06054804474115372, -0.052175965160131454, -0.0414687804877758, 0.025226064026355743, 0.04294651746749878, 0.04720638319849968, 0.040196843445301056, -0.06707590818405151, 0.01688171736896038, 0.13124725222587585, 0.06870204955339432, 0.023616470396518707, 0.08918488025665283, 0.004599261097609997, -0.03571508452296257, -0.052998729050159454, 0.031709279865026474, 0.008779587224125862, 0.0006242452654987574, -0.058398280292

# Menampilkan hasil perhitungan similarity antara query dan data sales order

In [93]:
similarity = np.dot(
    embeded_query,
    embeded_so_data
) / (
    np.linalg.norm(embeded_query) *
    np.linalg.norm(embeded_so_data)
)
print(f"Nilai query yang telah di embedding: \n{embeded_query}")
print(f"\nNilai data sales order yang telah di embedding: \n{embeded_so_data}")
print("\nNilai similarity antara query dan data sales order: ")
print(similarity)

Nilai query yang telah di embedding: 
[-0.10211622714996338, 0.02517821453511715, -0.039129339158535004, -0.04913618415594101, -0.11129520833492279, -0.027086937800049782, 0.04770001769065857, 0.016120124608278275, 0.028990212827920914, -0.018244776874780655, 0.11045758426189423, 0.04905102401971817, -0.03385816887021065, -0.1092018112540245, -0.03643539920449257, -0.06257018446922302, -0.042251162230968475, -0.08245939016342163, -0.05062417313456535, 0.0070544336922466755, 0.0019382309401407838, 0.03141807019710541, -0.06054804474115372, -0.052175965160131454, -0.0414687804877758, 0.025226064026355743, 0.04294651746749878, 0.04720638319849968, 0.040196843445301056, -0.06707590818405151, 0.01688171736896038, 0.13124725222587585, 0.06870204955339432, 0.023616470396518707, 0.08918488025665283, 0.004599261097609997, -0.03571508452296257, -0.052998729050159454, 0.031709279865026474, 0.008779587224125862, 0.0006242452654987574, -0.058398280292749405, 0.0009045260376296937, -0.02839969284832

# Simulasi Hasil Sistem retrival

In [129]:
# tahap embedding data perusahaan, dan penyimpanan ke vectorstore
vectorstore = FAISS.from_documents(so_data_as_document, embedding_model)

# mengatur bahwa nanti sistem akan mengambil K data yang paling cocok saat ada pertanyaan
result = vectorstore.similarity_search_with_relevance_scores(
    query="Tampilkan nama dan alamat customer, total amount, nilai PPN, dan grand amount, pada sales order dengan No. TRX SO: SO-BR0011-2605-00000006",
    k=len(so_data_as_document)
)

result

[(Document(id='f19d04e2-45e5-4cb7-aff1-01fc85c87e68', metadata={}, page_content='Business Unit: PENERBITAN, Sub Business Unit: Publishing JPBOOK, Cabang: JPBOOK Surabaya, No. TRX SO: SO-BR0011-2605-00000006, SO Date: 13/05/2026, SO Type: PUBLISHING, Item Type: FINISH GOOD, Customer Type: KOMERSIAL (B2B), Pemasaran: DIRECT (TANPA PERANTARA), Pembayaran: TEMPO, Customer: UPT SD N 01 GUNUNG CAHAYA, Alamat Customer: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung, No. NPWP: None, Nama NPWP: UPT SD N 01 GUNUNG CAHAYA, Contact Person: None, No. PO Customer: None, Bill To: UPT SD N 01 GUNUNG CAHAYA, Bill To Address: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung , Order Type 1: ONLINE, Order Type 2: SIPLAH, PPN Type: EXCLUDE, SO Prospek: False, Nama Penerima: UPT SD N 01 GUNUNG CAHAYA, Alamat: Gunung Cahya Desa Gunung Cahya Kec. Pakuan Ratu Kab. Way Kanan Prov. Lampung , Due Date: 27/05/2026, Payment Term: 120 HARI, Project: None, Dow